In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="qwen3:4b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking fictional information about Lunapolis, the capital of the moon, including its weather, cheese miner population, and union strike predictions.\n\n## SUMMARY\n- The capital of the moon is Lunapolis.\n- Weather in Lunapolis: clear skies, temperature range of 120°C to -100°C.\n- Population of cheese miners in Lunapolis: 100,000.\n- The cheese miners' union is predicted to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='0a903bc7-5e29-43b1-9430-419dc95a4172'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='71c62315-3a02-4c0e-adbe-b27067e61153'),
              AIMessage(content='As Lunapolis\' new president (and a form

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking fictional information about Lunapolis, the capital of the moon, including its weather, cheese miner population, and union strike predictions.

## SUMMARY
- The capital of the moon is Lunapolis.
- Weather in Lunapolis: clear skies, temperature range of 120°C to -100°C.
- Population of cheese miners in Lunapolis: 100,000.
- The cheese miners' union is predicted to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='6978e2bc-94f5-4366-8aff-235c5f0c54e0'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='667ac1b3-6591-4d35-89bc-0673030df981', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='1ad2334d-da63-4646-bb07-3e7737858806'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='32e3c646-f871-4a89-964b-dc0ff3e9972b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='dabc71ee-6080-4eec-92b6-41612f5ffcae'),
              AIMessage(content='I see — your question about temperature is helpful, but let\'s

In [8]:
print(response["messages"][-1].content)

I see — your question about temperature is helpful, but let's clarify things step by step to avoid confusion. **This is likely a misunderstanding about what "turned on" means**, so I’ll reorient the troubleshooting to be clear and actionable:

---

### 🔍 Let’s fix the confusion first (this is critical):
When you say *"it's plugged in and turned on"*, **this is confusing because:**
- **"Plugged in"** = The device is physically connected to a power source (e.g., charger, outlet).
- **"Turned on"** = The device is *actively running* (e.g., screen亮, lights blinking, working).

**If it’s *plugged in* but *not turning on* (i.e., not running), that’s the real problem.**  
*(Your device is probably plugged in but not powering up — which is why you’re asking about temperature.)*

✅ **Your device is likely:  
"Plugged in ✅" → BUT "Not turning on ❌"**

---

### 🛠️ Let’s troubleshoot *correctly* (starting with the easiest checks):
1️⃣ **Check for ANY lights or indicators** (this is the MOST import